# Meandering channel morphometric analysis

This code computes sinuous channel geomorphic parameters including meander wavelength, amplitude, and sinuosity, according to Leopold and Wolman (1957) methodology (see also Sylvester and Pirmez, 2017; Lemay et al., 2020).

### author:
Martin LEMAY
martin.lemay@mines-paris.org

### Related publication

If you use this script, please refer to the following publication:
- Lemay, M. (2026). pyBenD: Python Bend Dynamics (v1.0.3). Zenodo. https://doi.org/10.5281/zenodo.18451972 
- Lemay, M., Grimaud, J. L., Cojan, I., Rivoirard, J., & Ors, F. (2020). Geomorphic variability of submarine channelized systems along continental margins: Comparison with fluvial meandering channels. Marine and Petroleum Geology, 115, 104295. https://doi.org/10.1016/j.marpetgeo.2020.104295

*WARNING: results may slightly differ from the related paper because of subsequent modifications of the code.*

### Dependencies
- numpy
- scipy
- matplotlib
- pandas
- shapely

#### Python modules import

In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

parent_dir_path = os.path.dirname(os.getcwd())
package_path = os.path.join(parent_dir_path, "src")
if package_path not in sys.path:
    sys.path.append(package_path)
    
import pybend.algorithms.centerline_process_functions as cpf
import pybend.algorithms.geometry_functions as gf
import pybend.algorithms.plot_functions as plot
import pybend.io as io
from pybend.model.Centerline import Centerline
from pybend.model.Morphometry import Morphometry


#### Computation script

Compute channel geomorphic parameters according to the following methodology (see Lemay et al., 2020):  

1 resample centerline points; 

2 smooth centerline path; 

3 compute and smooth curvatures; 

4 compute inflection and apex points and define bends; 

5 compute bend geometric parameters

#### Input parameters

In [ ]:
working_dir = str(os.path.join(os.getcwd(), "data/"))
input_file_name = "centerline_Abreu_et_al_2003.csv" # centerline data file name - .csv file
output_file_name = input_file_name[:-4] + "_morphometry.csv" # morphometry measurements results file name
delimiter = ';' # column delimiter used in the input and output files
flip = False # to flip data if upstream and downstream end are reversed in the input file

# *** Channel centerline analysis parameters ***
smooth_distance = 1000 # [m] smoothing window applied when smoothing centerline path and curvatures (usually around twice channel width)
spacing = 50 # [m] spacing between channel points after resampling
use_fix_nb_points = False
window_curvature_filter = 15
sinuo_thres = 1.05 # minimum sinuosity allowed to define meanders
n = 2.

# channel dimension parameters used to fit computation windows 
width = 200# [m] -99999 / channel width; if -99999, use the default width of 800 m is taken (corresponding to median values of turbidites channels - see Lemay et al., 2020)
depth = 15 # [m] -99999 / channel mean bankfull depth; if -99999, use the default depth of 27 m is taken (corresponding to median values of turbidites channels - see Lemay et al., 2020)


assert width != -99999, "WARNING: no data for the width"
assert depth != -99999, "WARNING: no data for the depth"

# data loading from csv file   
if input_file_name[-3:] != "csv":
    sys.exit("ERROR: file format %s not managed"%(input_file_name[-3:]))

age, dataset = io.load_centerline_from_file(
    working_dir + input_file_name, kind=io.CenterlineIOFormat.CSV,
    x_prop="Cart_abscissa", y_prop="Cart_ordinate", delimiter=';'
)

# flip data
if flip:
    dataset = dataset.iloc[::-1].reset_index(drop=True)

# centerline object and meanders definition
#centerline = Centerline(0, data, spacing, smooth_distance, lag, nb, sinuo_thres)
centerline = Centerline(
                0,
                dataset,
                spacing,
                smooth_distance,
                use_fix_nb_points,
                window_curvature_filter,
                sinuo_thres,
                n,
                compute_curvature=True,
                interpol_props=True,
                find_bends=True,
            )
            
print("Centerline loaded")
print(f"Number of bends: {centerline.get_nb_bends()}")

#### Plot results

In [ ]:
domain=[[],[]]          # plotting area ((xmin, xmax), (ymin, ymax)) - use default pyplot values if the lists are empty 
plot_apex = True        # plot apex points
plot_inflex = True      # plot inflection points
plot_middle = False     # plot middle of inflection points
plot_centroid = False   # plot the centroid of the bend
plot_normal = False     # plot the normal to the channel at each channel point
scale_normal = 1.       # if plot_normal is True, scaling parameter for arraows

annotate = False     # write bend id
annot_text_size = 10 # annotation text size
color_bend = True    # color bend path in blue and red according to curvature
linewidth = 1        # centerline width
markersize = 2       # apex, inflection, middle point marker size

plot_filename = input_file_name[:-4] + ".png" # to save plot figure

fig, ax = plt.subplots(figsize=(5,5), dpi=150)   
plot.plot_bends(ax, (centerline.cl_points,), centerline.bends, domain = domain, annotate = annotate,
           plot_apex = plot_apex, plot_inflex = plot_inflex, plot_middle = plot_middle,
           plot_centroid = plot_centroid, plot_normal = plot_normal, scale_normal = scale_normal,
           annot_text_size = annot_text_size, color_bend = color_bend, 
           alpha=1, cl_color=False)

if len(domain[0]) > 0:
    plt.xlim(domain[0])
if len(domain[1]) > 0:
    plt.ylim(domain[1])
if len(domain[0])+len(domain[1]) == 0:
    plt.axis('equal')

plt.xlabel('X (m)')
plt.ylabel('Y (m)')

#plt.savefig(working_dir + plot_filename, dpi = 150)
plt.show()

In [ ]:
# compute average meander geomorphic parameters according to window_size
factor = 5. # estimated wavelength to width ratio; 5 is the median ratio for turbidites channels - see Lemay et al. (2020)

window_size = factor*width
leopold = True

morpho = Morphometry(centerline)
metrics = morpho.compute_bends_morphometry(False)
sinuo_window = [morpho.compute_bend_sinuosity_moving_window(i, window_size) for i in range(centerline.get_nb_bends())]
metric_averaged = pd.DataFrame(np.full_like(metrics, np.nan), columns=metrics.columns)
for i in range(centerline.get_nb_bends()):
    metric_averaged.loc[i] = morpho.compute_average_metric_window(i, window_size)

print(input_file_name)
print("System-averaged bend geometry")
print(metrics.mean(axis=0))
print()

print("Window-averaged meander geometry")
for bend in centerline.bends:    
    print("  Bend:", bend.id)
    print(f"Sinuosity (Window)       {sinuo_window[bend.id]}")
    print(metric_averaged.loc[bend.id])
    print()

print("Individual bend geometry")
for bend in centerline.bends:    
    print("  Bend:", bend.id)
    print(metrics.loc[bend.id])
    print()